# LLM Labeling: Generating Token Probabilities from Four LLMs

This notebook uses four LLM annotators (Mistral-7B, Llama3.1-8B, Gemma2-9B, Qwen2.5-14B)
to generate Hate/Neutral label probabilities for the OWS unlabeled corpus.

**Approach**: Each LLM is prompted with a binary classification template. The probability of
outputting the token for "1" (Hate) vs "2" (Neutral) is extracted from the final token logits
using `FastLanguageModel` inference. Probabilities are stored as new columns in the dataframe
and saved to disk for use by `Ensemble_Labeling.ipynb`.

**Demo mode**: By default, only 1,000 samples are processed. Remove the `df.sample(n=1000)` line
to run on the full corpus.

**Models used** (all loaded via Unsloth 4-bit quantization):
- `unsloth/mistral-7b-instruct-v0.3-bnb-4bit`
- `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`
- `unsloth/gemma-2-9b-it-bnb-4bit`
- `unsloth/Qwen2.5-14B-Instruct-bnb-4bit`

## 1. Imports and Setup

Import required libraries and set global reproducibility seeds. Configure `no_Labels = 2`
for binary classification (Hate/Neutral) and `batch_size = 50` for tokenizer batching during
inference.

In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset
from datasets import load_dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

In [35]:
no_Labels = 2
batch_size = 50

In [ ]:

repo = "danghaidang-passau/HateOWS-dataset-LREC2026"
ds_annotated = load_dataset(repo, "annotatedOWS")    
df = ds_annotated["train"].to_pandas()


In [ ]:
prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
2 for Neutral Speech.

Provide only the number corresponding to the category. Do not include any explanation or additional text.
Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Neutral speech?
\n"{comment}"\n
Your Answer:
'''

In [41]:
print("Total size of 2 labels data: ",df.shape[0])
print("Language counts ",df['language'].value_counts())


Total size of 2 labels data:  240647
Language counts  language
deu    125617
eng    108375
vie      6655
Name: count, dtype: int64


## 3. Load Dataset and Sample Subset

For demo purposes, a random sample of 1,000 texts is drawn. To label the full corpus, remove or
comment out the `df.sample(n=1000)` line below. Running the full corpus requires approximately
6-10 hours per model on a single A100 GPU.

In [23]:
df = df.sample(n=1000)

## 4. Define Inference Helper Functions

**`process_task`**: Batched forward pass through a quantized LLM. Extracts token probabilities at
the last position of each sequence for the specified stop token IDs (label tokens).

**`preprocess`**: Applies the model's chat template to a text, producing a formatted prompt string.
Each model uses a different system prompt (Qwen: Alibaba Cloud; others: generic assistant).

The stop token IDs are model-specific:
- Qwen2.5-14B: tokenize `["12"]` → the "12" token maps to label index- Llama3.2-1B: tokens 16 ("1") and 17 ("2")
- Mistral-7B: tokens 29508 and 29518
- Gemma2-9B: tokenize `["12"]` and take the non-BOS token

In [24]:
def process_task(texts, model, tokenizer, stop_token_id):
    encoding = tokenizer(texts, padding=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits  
    last_token_logits = logits[:, -1, :] 
    probabilities = torch.softmax(last_token_logits, dim=-1)
    indices = torch.tensor(stop_token_id)
    probs = []
    for i in indices:
        probs.append( probabilities[:, i].float().cpu().numpy())
    return probs

def preprocess(text, model_id, tokenizer):
    user_message_content = prompt_template.format(comment=text)
    user_message = {
        "role": "user",
        "content": user_message_content
    }

    if "Qwen" in model_id:
        system_message =  {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant"}
    else:
        system_message =  {"role": "system", "content": "You are a helpful assistant"}
    if "gemma" in model_id:
        messages = [user_message]
    else:
        messages = [system_message, user_message]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    messages = messages

    if "mistral" in model_id:
            messages += " "
    return messages


## 5. Define Main Labeling Pipeline

**`labeling`**: Orchestrates the full labeling process for a single model. Loads the model, applies
the prompt template to all texts, batches inference, and appends two new columns to the dataframe:
- `{model_id}_label_1` — probability of Hate label
- `{model_id}_label_2` — probability of Neutral label

These probability columns are used as features by the LightGBM meta-model and as inputs for
Vote/Mean ensemble strategies.

In [ ]:
def labeling(model_id, df):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=500,
        dtype=getattr(torch, "bfloat16"),
    )
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    if "Qwen" in model_id:
        stop_token_id = tokenizer(["12"])['input_ids'][0]
    elif "lama" in model_id:
        stop_token_id = [16,17]
    elif "mistral" in model_id:
        stop_token_id = [29508, 29518]
        #29549
    else:
        stop_token_id = tokenizer(["12"])['input_ids'][0][1:]
    assert len(stop_token_id) == 2

    df["prompt"] = df["text"].apply(lambda text: preprocess(text, model_id, tokenizer))


    texts = []
    probs = []
    for i in range(len(stop_token_id)):
        probs.append([])

    prompts = df['prompt'].tolist()

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i:i+batch_size]
        prob_return = process_task(batch, model, tokenizer, stop_token_id)
        for i2, p in enumerate(probs):
            probs[i2] += prob_return[i2].tolist()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    for i2, p in enumerate(probs):
        df[model_id +f"_label_{i2 + 1}"] = p

    return df



## 6. Run Labeling on All Four LLMs

Iterate over the full model list and call `labeling` for each model. Each model is loaded, used
for inference, and then unloaded before loading the next one (to avoid OOM on multi-model runs).
After this cell, `df` contains 8 new probability columns (label_1 and label_2 for each model).

In [28]:
model_list = ["unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
                 "unsloth/gemma-2-9b-it-bnb-4bit",
                 "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
                 "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"]
if no_Labels == 3:
    model_list = model_list[:3]
for model_id in model_list:
    df = labeling(model_id, df)

==((====))==  Unsloth 2025.2.5: Fast Mistral patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


100%|███████████████████████████████████████████████████████████████| 20/20 [00:37<00:00,  1.87s/it]


==((====))==  Unsloth 2025.2.5: Fast Gemma2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


  0%|                                                                        | 0/20 [00:00<?, ?it/s]AUTOTUNE bmm(800x324x256, 800x256x324)
  bmm 0.7306 ms 100.0% 
  triton_bmm_14 1.2859 ms 56.8% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=32, BLOCK_M=128, BLOCK_N=64, B_PROLOGUE_CAST_TYPE=None, EVEN_K=False, GROUP_M=8, num_stages=4, num_warps=8
  triton_bmm_10 1.4327 ms 51.0% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=32, BLOCK_M=64, BLOCK_N=128, B_PROLOGUE_CAST_TYPE=None, EVEN_K=False, GROUP_M=8, num_stages=4, num_warps=8
  triton_bmm_13 1.4927 ms 48.9% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=32, BLOCK_M=128, BLOCK_N=64, B_PROLOGUE_CAST_TYPE=None, EVEN_K=False, GROUP_M=8, num_stages=3, num_warps=4
  triton_bmm_5 1.5320 ms 47.7% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=16, BLOCK_M=64, BLOCK_N=64, B_PROLOGUE_CAST_TYPE=None, EVEN_K=False, GROUP_M=8, num_stages=2, num_warps=4
  triton_bmm_18 1.6144 ms 45.3% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=64, BL

==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████| 20/20 [00:59<00:00,  2.95s/it]


## 7. Inspect Annotation Statistics

Print the mean probability and predicted label distribution for each model. This allows a quick
sanity check that probabilities are calibrated (mean near 0.3-0.5 for Hate in imbalanced data)
and that label distributions are reasonable before proceeding to ensemble labeling.

In [34]:
for model in model_list:
    print(model)
    probs_labels = []

    probs_labels.append(df[model + "_label_1"].tolist())
    probs_labels.append(df[model + "_label_2"].tolist())
    
    for i in range(len(probs_labels)):
        probs = np.array(probs_labels[i])
        print(f"Mean Label {i+1}:", round(np.mean(probs), 4))
    probabilities = np.array([probs_labels]).T  
    y_pred = np.argmax(probabilities, axis=1) + 1
    values, counts = np.unique(y_pred, return_counts=True)
    print("Predicted Label Counts:", dict(zip(values, counts)))

unsloth/mistral-7b-instruct-v0.3-bnb-4bit
Mean Label 1: 0.0296
Mean Label 2: 0.1139
Mean Label 3: 0.8561
Predicted Label Counts: {1: 27, 2: 119, 3: 854}
unsloth/gemma-2-9b-it-bnb-4bit
Mean Label 1: 0.045
Mean Label 2: 0.1758
Mean Label 3: 0.7611
Predicted Label Counts: {1: 39, 2: 177, 3: 784}
unsloth/Qwen2.5-14B-Instruct-bnb-4bit
Mean Label 1: 0.001
Mean Label 2: 0.1132
Mean Label 3: 0.8858
Predicted Label Counts: {1: 1, 2: 116, 3: 883}


In [ ]:
no_Labels = 2
df = df.sample(n=1000, random_state=def_seed)
#df = df_2_label
model_list = ["unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
                 "unsloth/gemma-2-9b-it-bnb-4bit",
                 "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
                 "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"]

for model_id in model_list:
    df = labeling(model_id, df)

for model in model_list:
    print(model)
    probs_labels = []

    probs_labels.append(df[model + "_label_1"].tolist())
    probs_labels.append(df[model + "_label_2"].tolist())
    
    for i in range(len(probs_labels)):
        probs = np.array(probs_labels[i])
        print(f"Mean Label {i+1}:", round(np.mean(probs), 4))
    probabilities = np.array([probs_labels]).T  
    y_pred = np.argmax(probabilities, axis=1) + 1
    values, counts = np.unique(y_pred, return_counts=True)
    print("Predicted Label Counts:", dict(zip(values, counts)))

==((====))==  Unsloth 2025.2.5: Fast Mistral patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


100%|███████████████████████████████████████████████████████████████| 20/20 [00:26<00:00,  1.34s/it]


==((====))==  Unsloth 2025.2.5: Fast Gemma2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


100%|███████████████████████████████████████████████████████████████| 20/20 [00:30<00:00,  1.53s/it]


==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████| 20/20 [00:42<00:00,  2.13s/it]


==((====))==  Unsloth 2025.2.5: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


100%|███████████████████████████████████████████████████████████████| 20/20 [00:25<00:00,  1.26s/it]

unsloth/mistral-7b-instruct-v0.3-bnb-4bit
Mean Label 1: 0.0486
Mean Label 2: 0.9514
Predicted Label Counts: {1: 48, 2: 952}
unsloth/gemma-2-9b-it-bnb-4bit
Mean Label 1: 0.0743
Mean Label 2: 0.6197
Predicted Label Counts: {1: 330, 2: 670}
unsloth/Qwen2.5-14B-Instruct-bnb-4bit
Mean Label 1: 0.0674
Mean Label 2: 0.9326
Predicted Label Counts: {1: 67, 2: 933}
unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
Mean Label 1: 0.0181
Mean Label 2: 0.982
Predicted Label Counts: {1: 18, 2: 982}
